# Milestone 1: dataset preparation, text embeddings and first samples (Google Colab)

Runtime → Change runtime type → **T4 GPU** (or better). Data and outputs are kept on Google Drive in `MyDrive/IE7615_Project2`, so a disconnected session can resume.

| Step | Script | Output |
|---|---|---|
| 1 | `01_prepare_data.py` | COCO val2017 subset, 512×512 images, `results/dataset_stats.json` |
| 2 | `02_encode_captions.py` | CLIP ViT-L/14 embeddings for every caption |
| 3 | `03_embedding_checks.py` | caption similarity + text→image retrieval, `results/embedding_checks.json` |
| 4 | `04_sample_prompts.py` | 5 prompt→image samples, external-embedding conditioning check |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/IE7615_Project2'
except Exception as e:
    print('Google Drive not mounted, outputs stay in /content/outputs:', e)
    DRIVE = '/content/outputs'
REPO = 'https://github.com/anshuman-dataverse/text-to-image-generation.git'
NUM_IMAGES = 5000
os.makedirs(f'{DRIVE}/data', exist_ok=True)

In [ ]:
%cd /content
!test -d project || git clone -q $REPO project
# fetch + hard reset: Colab-generated results/ must not block updating to the latest code
!cd project && git remote set-url origin $REPO && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
!pip -q install -r project/requirements.txt
# Colab's preinstalled torchao is too old for the latest diffusers (FqnToConfig import error); we don't use it
!pip -q uninstall -y torchao
# keep the COCO download and processed data on Drive
!rm -rf project/milestone1/data && ln -sfn "$DRIVE/data" project/milestone1/data

## 1. Download COCO val2017, clean captions, subset and resize

In [ ]:
%cd /content/project/milestone1/scripts

import os, subprocess
from pathlib import Path
from IPython.display import Image, display

def run(script, *args):
    """Run a milestone script, stream its output here and to ../runs/<script>.log, stop on failure."""
    log = Path('../runs') / (Path(script).stem + '.log')
    log.parent.mkdir(exist_ok=True)
    commit = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
    print('code version:', commit)
    with open(log, 'w') as f:
        f.write(f'code version: {commit}\n')
        p = subprocess.Popen(['python', '-u', script, *map(str, args)], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='')
            f.write(line)
    if p.wait():
        raise RuntimeError(f'{script} failed (exit code {p.returncode}). The error is printed above and saved in {log}')

def show(*names):
    for n in names:
        path = Path('../screenshots') / n
        display(Image(str(path))) if path.exists() else print(f'missing: {path} (the script before this cell did not finish)')

In [ ]:
run('01_prepare_data.py', '--num-images', NUM_IMAGES)

In [ ]:
show('dataset_examples.png', 'caption_lengths.png')

## 2. Text → embedding pipeline (CLIP ViT-L/14)

In [ ]:
run('02_encode_captions.py')

In [ ]:
import sys, torch
sys.path.insert(0, '.')
from t2i.embeddings import TextEncoder
enc = TextEncoder('openai/clip-vit-large-patch14')
caption = 'A man riding a bicycle down a city street.'
tok = enc.tokenize([caption])
print(enc.tokenizer.convert_ids_to_tokens(tok['input_ids'][0][:int(tok['attention_mask'].sum())]))
out = enc.encode([caption])
print('per-token (U-Net cross-attention context):', tuple(out['hidden'].shape))
print('pooled (EOS token):', tuple(out['pooled'].shape))
del enc; torch.cuda.empty_cache()

## 3. Embedding sanity checks

In [ ]:
run('03_embedding_checks.py')

In [ ]:
show('caption_similarity.png', 'retrieval_examples.png')

## 4. Five prompt → image test runs (Stable Diffusion v1.5, DDIM, 30 steps, guidance 7.5)

In [ ]:
run('04_sample_prompts.py')

In [ ]:
show('milestone1_samples.png', 'conditioning_check.png')

## 5. Save outputs to Drive
Download `milestone1_outputs.zip`, unzip into `milestone1/` of the local repo and commit.

In [ ]:
%cd /content/project/milestone1
!zip -qr "$DRIVE/milestone1_outputs.zip" results screenshots runs
!ls -la "$DRIVE"